# Parkinson's Disease Voice Screening & Clinical Decision Support Platform
## Notebook 02: Audio Preprocessing, Normalization & Fixed Segmentation

> **DISCLAIMER:** This software is a research screening tool, **NOT** a diagnostic device.

### Purpose & Architecture Alignment
This notebook implements standardized audio preprocessing to prepare raw voice recordings for downstream feature extraction (frozen WavLM-Base-Plus self-supervised representations).

To guarantee **100% train/inference parity**, all preprocessing logic is encapsulated in the standalone, framework-agnostic module:
`ml/preprocessing/audio_preprocessing.py`
and its versioned configuration file:
`ml/preprocessing/preprocess_config.json`

Both the Colab training pipeline and the local FastAPI production backend import this exact module and configuration verbatim.

### Preprocessing Stages:
1. **Mono Channel Reduction:** Averages stereo/multichannel tracks into a single acoustic channel.
2. **Standard Resampling (16 kHz):** Converts heterogeneous sampling rates (16 kHz, 44.1 kHz) to the exact 16,000 Hz input requirement of WavLM.
3. **Leading/Trailing Silence Trimming:** Removes ambient silence below 30 dB relative to peak energy (`top_db=30`) using `librosa.effects.trim`.
4. **Peak Normalization:** Scales peak amplitude to $[-1.0, 1.0]$.
5. **Fixed-Length Segmentation with Repeat-Padding:** Pads or crops the trimmed waveform to exactly $4.0\text{ s} \times 16{,}000\text{ Hz} = 64{,}000\text{ samples}$. Short recordings use **repeat-padding** rather than zero-padding, preventing dilution of self-supervised acoustic representations.


In [1]:
# Cell 1: Install & import required libraries
import sys
import subprocess

required_packages = ["soundfile", "librosa", "pandas", "numpy", "matplotlib", "tqdm"]
for pkg in required_packages:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from tqdm import tqdm

print("All dependencies successfully imported.")


All dependencies successfully imported.


In [2]:
# Cell 2: Resolve paths and import standalone preprocessing module
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

# Add repository root to sys.path so ml.preprocessing can be imported identically to production
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.preprocessing.audio_preprocessing import PreprocessConfig, preprocess_audio

CONFIG_PATH = PROJECT_ROOT / "ml" / "preprocessing" / "preprocess_config.json"
METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "metadata.csv"
AUDIO_OUT_DIR = PROJECT_ROOT / "data" / "processed" / "audio"

# Load versioned configuration
config = PreprocessConfig.from_json(CONFIG_PATH)
print("=== PREPROCESSING CONFIGURATION ===")
print(f"  Target Sample Rate: {config.target_sr} Hz")
print(f"  Trim Silence Threshold: {config.trim_top_db} dB")
print(f"  Peak Normalize:     {config.normalize}")
print(f"  Segment Duration:   {config.segment_seconds} seconds")
print(f"  Target Samples:     {config.target_samples} samples")
print(f"  Pad Mode:           {config.pad_mode} (repeat-pad)")
print("====================================")


=== PREPROCESSING CONFIGURATION ===
  Target Sample Rate: 16000 Hz
  Trim Silence Threshold: 30 dB
  Peak Normalize:     True
  Segment Duration:   4.0 seconds
  Target Samples:     64000 samples
  Pad Mode:           repeat (repeat-pad)


In [3]:
# Cell 3: Apply preprocessing pipeline to all files in metadata.csv
df = pd.read_csv(METADATA_PATH)
print(f"Loaded metadata table: {len(df)} recordings across {df['subject_id'].nunique()} subjects.")

processed_records = []
AUDIO_OUT_DIR.mkdir(parents=True, exist_ok=True)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing Audio"):
    raw_rel_path = row["file_path"]
    split = row["split"]
    subject_id = row["subject_id"]
    raw_path = PROJECT_ROOT / raw_rel_path
    fname = raw_path.name

    # Define target path: data/processed/audio/{split}/{subject_id}/{original_filename}
    target_dir = AUDIO_OUT_DIR / split / subject_id
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / fname
    target_rel_path = str(target_path.relative_to(PROJECT_ROOT))

    # Read raw audio
    waveform, sr = sf.read(str(raw_path))

    # Execute canonical preprocessing function
    proc_waveform = preprocess_audio(waveform, sr, config)

    # Save processed audio file (16-bit PCM WAV)
    sf.write(str(target_path), proc_waveform, config.target_sr, subtype="PCM_16")

    processed_records.append(target_rel_path)

# Update metadata table with processed_path
df["processed_path"] = processed_records
df.to_csv(METADATA_PATH, index=False)
print(f"\nSuccessfully processed {len(df)} files.")
print(f"Updated metadata table written to: {METADATA_PATH}")


Loaded metadata table: 831 recordings across 65 subjects.
Preprocessing Audio: 100%|██████████| 831/831 [01:58<00:00,  7.03it/s]

Successfully processed 831 files.
Updated metadata table written to: /Users/eshwarsaielugam/Documents/prototype/data/processed/metadata.csv


In [4]:
# Cell 4: Visual verification & signal sanity check (5 random files per class)
# Verify trimming/padding preserves vocal signals without clipping or distortion
np.random.seed(42)
healthy_sample = df[df["label"] == 0].sample(5, random_state=42)
pd_sample = df[df["label"] == 1].sample(5, random_state=42)
spot_samples = pd.concat([healthy_sample, pd_sample]).reset_index(drop=True)

print("=== SPOT-CHECK SIGNAL COMPARISON (5 HEALTHY + 5 PARKINSON'S) ===")
fig, axes = plt.subplots(10, 2, figsize=(15, 20))
plt.subplots_adjust(hspace=0.6, wspace=0.3)

for i, (_, row) in enumerate(spot_samples.iterrows()):
    raw_path = PROJECT_ROOT / row["file_path"]
    proc_path = PROJECT_ROOT / row["processed_path"]

    raw_audio, raw_sr = sf.read(str(raw_path))
    proc_audio, proc_sr = sf.read(str(proc_path))

    if raw_audio.ndim > 1:
        raw_audio = np.mean(raw_audio, axis=1)

    raw_dur = len(raw_audio) / raw_sr
    proc_dur = len(proc_audio) / proc_sr
    class_name = "Healthy" if row["label"] == 0 else "Parkinson's"

    print(f"[{i+1:02d}/10] {class_name:12s} | Subj: {row['subject_id']:22s} | "
          f"Raw: {raw_dur:6.2f}s @ {raw_sr:5d}Hz | "
          f"Proc: {proc_dur:4.2f}s ({len(proc_audio)} smp) @ {proc_sr}Hz | "
          f"Peak: {np.max(np.abs(proc_audio)):.2f}")

    # Plot raw waveform
    raw_t = np.linspace(0, raw_dur, len(raw_audio))
    axes[i, 0].plot(raw_t, raw_audio, color="royalblue", lw=0.6)
    axes[i, 0].set_title(f"{class_name} Raw: {row['subject_id']} ({raw_dur:.1f}s, {raw_sr}Hz)", fontsize=9)
    axes[i, 0].set_xlabel("Time (s)", fontsize=8)
    axes[i, 0].set_ylabel("Amp", fontsize=8)
    axes[i, 0].grid(True, alpha=0.3)

    # Plot processed waveform
    proc_t = np.linspace(0, proc_dur, len(proc_audio))
    axes[i, 1].plot(proc_t, proc_audio, color="darkorange", lw=0.6)
    axes[i, 1].set_title(f"Processed: 4.0s Window @ 16kHz (Peak={np.max(np.abs(proc_audio)):.2f})", fontsize=9)
    axes[i, 1].set_xlabel("Time (s)", fontsize=8)
    axes[i, 1].set_ylabel("Norm Amp", fontsize=8)
    axes[i, 1].grid(True, alpha=0.3)

plt.show()


=== SPOT-CHECK SIGNAL COMPARISON (5 HEALTHY + 5 PARKINSON'S) ===
[01/10] Healthy      | Subj: ehc_ANGELA_G           | Raw:  73.53s @ 16000Hz | Proc: 4.00s (64000 smp) @ 16000Hz | Peak: 1.00
[02/10] Healthy      | Subj: ehc_VITANTONIO_D       | Raw:   9.25s @ 16000Hz | Proc: 4.00s (64000 smp) @ 16000Hz | Peak: 1.00
[03/10] Healthy      | Subj: ehc_PORCELLI_A         | Raw:   6.53s @ 16000Hz | Proc: 4.00s (64000 smp) @ 16000Hz | Peak: 1.00
[04/10] Healthy      | Subj: ehc_AGNESE_P           | Raw:   6.50s @ 16000Hz | Proc: 4.00s (64000 smp) @ 16000Hz | Peak: 1.00
[05/10] Healthy      | Subj: ehc_VITO_A             | Raw:  10.01s @ 16000Hz | Proc: 4.00s (64000 smp) @ 16000Hz | Peak: 1.00
[06/10] Parkinson's  | Subj: pd_17-28_Nicolò_C      | Raw:   9.69s @ 16000Hz | Proc: 4.00s (64000 smp) @ 16000Hz | Peak: 1.00
[07/10] Parkinson's  | Subj: pd_1-5_Domenico_C      | Raw:  19.10s @ 44100Hz | Proc: 4.00s (64000 smp) @ 16000Hz | Peak: 1.00
[08/10] Parkinson's  | Subj: pd_17-28_Mario_B       |

In [5]:
# Cell 5: Quality Assurance Checklist & Data Contract Validation
print("=== PREPROCESSING VALIDATION CHECKLIST ===")

# 1. Verify processed_path column existence and integrity
assert "processed_path" in df.columns, "Column 'processed_path' missing from metadata.csv"
assert df["processed_path"].isna().sum() == 0, "Null values detected in 'processed_path'"
print(f"✓ Metadata contract verified: 'processed_path' column present on all {len(df)} rows.")

# 2. Verify all files exist on disk
missing_files = []
for p in df["processed_path"]:
    if not (PROJECT_ROOT / p).exists():
        missing_files.append(p)
assert len(missing_files) == 0, f"Missing {len(missing_files)} processed files: {missing_files[:5]}"
print(f"✓ File presence verified: All {len(df)} processed audio files exist on disk.")

# 3. Verify exact length and sample rate across all processed files
length_mismatches = []
sr_mismatches = []
for p in tqdm(df["processed_path"], desc="Auditing Processed Audio"):
    info = sf.info(str(PROJECT_ROOT / p))
    if info.frames != config.target_samples:
        length_mismatches.append((p, info.frames))
    if info.samplerate != config.target_sr:
        sr_mismatches.append((p, info.samplerate))

assert len(length_mismatches) == 0, f"Frame length mismatches found: {length_mismatches[:5]}"
print(f"✓ Length assertion passed: Every file has exactly {config.target_samples} frames ({config.segment_seconds}s).")

assert len(sr_mismatches) == 0, f"Sample rate mismatches found: {sr_mismatches[:5]}"
print(f"✓ Sample rate assertion passed: Every file is exactly {config.target_sr} Hz.")

# 4. Verify subject and split columns remain intact
assert "subject_id" in df.columns and df["subject_id"].nunique() == 65, "Subject IDs altered or lost"
assert set(df["split"].unique()) == {"train", "val", "test"}, "Splits altered or lost"
print("✓ Split and Subject ID integrity preserved.")

print("\nSTATUS: ALL PREPROCESSING VALIDATION CHECKS PASSED.")


=== PREPROCESSING VALIDATION CHECKLIST ===
✓ Metadata contract verified: 'processed_path' column present on all 831 rows.
✓ File presence verified: All 831 processed audio files exist on disk.
Auditing Processed Audio: 100%|██████████| 831/831 [00:00<00:00, 1842.21it/s]
✓ Length assertion passed: Every file has exactly 64000 frames (4.0s).
✓ Sample rate assertion passed: Every file is exactly 16000 Hz.
✓ Split and Subject ID integrity preserved.

STATUS: ALL PREPROCESSING VALIDATION CHECKS PASSED.
